## 課題
今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) をソフトマックス回帰によって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

    ### 目標値 (Target)
Accuracy: 80%

### ルール
- 訓練データは`x_train`， `y_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train、y_train`以外の学習データは使わないでください．**
- **隠れ層のない単層のモデルとして実装してください.**
- **ソフトマックス回帰のアルゴリズム部分の実装はnumpyのみで行ってください** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

In [4]:
# Library Import 
import pandas as pd
import numpy as np
import os
import sys

In [5]:
# モジュールを固定する。
# sys.module に None を入れておくと、以降 import tesorflow が ImportError になる。
# アルゴリズムは numpy のみというルールを抜けないようにしている。
sys.modules['tensorflow'] = None

In [6]:
# データの格納場所の指定
work_dir = '/Users/ryosuke/ai-engneering/data/jdla-e/lecture02/'

def load_fashionmnist():
    # ①データの読み込み
    # .npy は Numpy 配列を保存したファイル。読み込むと ndarray が返る。
    x_test = np.load(work_dir + 'x_test.npy')   # テスト画像（予測対象なので、正解ラベルは渡されない。）
    x_train = np.load(work_dir + 'x_train.npy') # 訓練画像
    y_train = np.load(work_dir + 'y_train.npy') # 訓練ラベル（0~9の整数)  

    # ②データの前処理
    # 画像: (N, 28, 28) -> (N, 784) に平坦化し、画素値 0~255 を 0~1 に正規化を行う。
    x_train = x_train.reshape(-1, 784).astype('float32') / 255
    x_test = x_test.reshape(-1, 784).astype('float32') / 255
    '''
    もともとMNISTの画像は以下のような形になっている。
    1枚の画像 = 28ピクセル * 28ピクセル
    つまり、1枚の画像は二次元の表です。
    [
      [0, 0, 12, ...],
      [0, 35, 200, ...],
      ...
    ]
    これをそのまま、計算すると大変になるので、1枚の画像を784(=28*28)のピクセルが並んだ１本のベクトルとして扱う。
    そのため、(N, 784)に平坦化している。　※Nは画像の枚数を指す。
    そして画像の各ピクセルは、通常 0〜255 の整数です。
    0 = 黒
    255 = 白
    このままだと数値が大きいので、機械学習ではよく 0〜1 に変換します。
    なので、/255 をしてあげることで、0~255の画素値が 0~1の間に収めることができるようになる。
    '''
    # ③ラベルをOne hot にする。
    y_train = np.eye(10)[y_train.astype('int32')]
    '''
    np.eye(10)は、10×10の単位行列を作る。
    [
     [1,0,0,0,0,0,0,0,0,0],
     [0,1,0,0,0,0,0,0,0,0],
     [0,0,1,0,0,0,0,0,0,0],
    ...
     [0,0,0,0,0,0,0,0,0,1]
    ]
    上記に対して、正解ラベルが'3'だった場合は、
    np.eye(10)[3]
    によって、[0,0,1,0,0,0,0,0,0,0]を取得することができる。
    最終的にソフトマックス関数のと正解ラベルが以下のようになった場合に、それぞれのクロスエントロピー誤差を計算する。
    正解：[0,0,0,1,0,0,0,0,0,0]
    予測：[0.01,0.02,0.03,0.80,0.01,0.02,0.01,0.05,0.03,0.02]
    '''  
    return x_train, y_train, x_test

In [7]:
x_train, y_train, x_test = load_fashionmnist()

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 検証データを分割
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train, y_train, test_size=0.1, random_state=42
)

In [8]:
# ===== ソフトマックス関数の実装 =====
def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [9]:
# ===== 重みの定義 =====
W = np.random.randn(784, 10)
b = np.zeros(10)

In [10]:
# best parameter保存用
best_W = W.copy()
best_b = b.copy()
best_acc = 0.0

In [12]:
def train_epoch(x, t, eps=0.1, batch_size=256):
    global W, b

    n_samples = x.shape[0]
    indices = np.random.permutation(n_samples)
    losses = []

    for start_idx in range(0, n_samples, batch_size):
        batch_idx = indices[start_idx:start_idx + batch_size]
        x_batch = x[batch_idx]
        t_batch = t[batch_idx]

        # forward
        logits = np.dot(x_batch, W) + b
        y = softmax(logits)

        # loss
        loss = -np.mean(np.sum(t_batch * np.log(y + 1e-7), axis=1))
        losses.append(loss)

        # backward
        delta = (y - t_batch) / x_batch.shape[0]
        grad_W = np.dot(x_batch.T, delta)
        grad_b = np.sum(delta, axis=0)

        # update
        W -= eps * grad_W
        b -= eps * grad_b

    return np.mean(losses)

In [14]:
def valid(x, t):
    logits = np.dot(x, W) + b
    y = softmax(logits)
    loss = -np.mean(np.sum(t * np.log(y + 1e-7), axis=1))
    acc = accuracy_score(t.argmax(axis=1), y.argmax(axis=1))
    return loss, acc

In [15]:
for epoch in range(100):
    # 学習率を途中で少し下げる
    if epoch < 50:
        lr = 0.1
    elif epoch < 80:
        lr = 0.05
    else:
        lr = 0.01

    train_loss = train_epoch(x_train, y_train, eps=lr, batch_size=256)
    valid_loss, valid_acc = valid(x_valid, y_valid)

    if valid_acc > best_acc:
        best_acc = valid_acc
        best_W = W.copy()
        best_b = b.copy()

    print(f"epoch: {epoch+1}, lr: {lr:.3f}, train_loss: {train_loss:.4f}, valid_loss: {valid_loss:.4f}, valid_acc: {valid_acc:.4f}")


epoch: 1, lr: 0.100, train_loss: 3.4797, valid_loss: 2.1752, valid_acc: 0.5867
epoch: 2, lr: 0.100, train_loss: 1.9189, valid_loss: 1.6960, valid_acc: 0.6528
epoch: 3, lr: 0.100, train_loss: 1.5787, valid_loss: 1.4791, valid_acc: 0.6795
epoch: 4, lr: 0.100, train_loss: 1.3997, valid_loss: 1.3426, valid_acc: 0.6953
epoch: 5, lr: 0.100, train_loss: 1.2824, valid_loss: 1.2490, valid_acc: 0.7145
epoch: 6, lr: 0.100, train_loss: 1.1987, valid_loss: 1.1837, valid_acc: 0.7232
epoch: 7, lr: 0.100, train_loss: 1.1367, valid_loss: 1.1211, valid_acc: 0.7333
epoch: 8, lr: 0.100, train_loss: 1.0849, valid_loss: 1.0818, valid_acc: 0.7373
epoch: 9, lr: 0.100, train_loss: 1.0441, valid_loss: 1.0360, valid_acc: 0.7458
epoch: 10, lr: 0.100, train_loss: 1.0078, valid_loss: 1.0071, valid_acc: 0.7518
epoch: 11, lr: 0.100, train_loss: 0.9768, valid_loss: 0.9868, valid_acc: 0.7590
epoch: 12, lr: 0.100, train_loss: 0.9513, valid_loss: 0.9529, valid_acc: 0.7570
epoch: 13, lr: 0.100, train_loss: 0.9270, valid_l

In [17]:
# best weights を使用
W = best_W
b = best_b

y_pred = softmax(np.dot(x_test, W) + b)

In [18]:
submission = pd.Series(y_pred.argmax(axis=1), name='label')
submission.to_csv(work_dir + '/output_02.csv',
                  header=True, index_label='id')